# Exploracion inicial de registros de reproduccion

Analisis exploratorio de registros de reproduccion de mi cuenta personal de TIDAL




In [4]:
import os
print(os.getcwd())

/workspaces/tidal-listening-analysis/notebooks


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('/workspaces/tidal-listening-analysis/data/raw/streaming.csv')

In [7]:
f_and_c = df.shape
c_names = df.columns

print(f"Total de reproducciones registradas: {f_and_c[0]}")
print(f"Total de columnas: {f_and_c[1]}")
for i, col in enumerate(df.columns):
    print(f"{i+1}. {col}")

Total de reproducciones registradas: 34403
Total de columnas: 14
1. artist_name
2. track_title
3. entry_date
4. stream_duration_ms
5. product_type
6. ip_address
7. client_name_from_session
8. country_name
9. region_name
10. city_name
11. postal_code
12. time_zone
13. album_id
14. album_title


In [11]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 34403 entries, 0 to 34402
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   artist_name               34402 non-null  str    
 1   track_title               34403 non-null  str    
 2   entry_date                34403 non-null  str    
 3   stream_duration_ms        34403 non-null  float64
 4   product_type              34403 non-null  str    
 5   ip_address                7579 non-null   str    
 6   client_name_from_session  33501 non-null  str    
 7   country_name              7579 non-null   str    
 8   region_name               7579 non-null   str    
 9   city_name                 7579 non-null   str    
 10  postal_code               7579 non-null   float64
 11  time_zone                 7579 non-null   str    
 12  album_id                  33261 non-null  float64
 13  album_title               33261 non-null  str    
dtypes: float64(3), st

Con 34403 registros, ninguna linea esta vasia, mas sin embargo, son valores nulos.
Once de los catorce registros son de tipo string, los restoantes son de tipo fotante (desimal).
Analisaremos mas antes de empezar a limiar.

In [41]:
faltantes = df.album_title.isna().sum()

print(faltantes)

1142


# ¿Cual es el % de valores faltantes por cada campo?

In [ ]:
porcen_f = df.isna().mean() * 100

print("'%' de registros faltantes por columna")
for i, (columna, valor) in enumerate(porcen_f.items(), 1):
    print(f"{i}. {columna}  {valor:.5f}%")

'%' de registros faltantes por columna
1. artist_name  0.00291%
2. track_title  0.00000%
3. entry_date  0.00000%
4. stream_duration_ms  0.00000%
5. product_type  0.00000%
6. ip_address  77.96994%
7. client_name_from_session  2.62186%
8. country_name  77.96994%
9. region_name  77.96994%
10. city_name  77.96994%
11. postal_code  77.96994%
12. time_zone  77.96994%
13. album_id  3.31948%
14. album_title  3.31948%


Hay tres observaciones que merecen atención.

Primera: las columnas relacionadas con ubicación e IP tienen aproximadamente un 78 % de valores faltantes. Es una proporción considerable, pero no necesariamente un error. Ya habíamos identificado que TIDAL modificó el registro de ciertos metadatos.

Segunda: album_id y album_title tienen exactamente el mismo porcentaje de valores faltantes: 3.31948 %. Esto sugiere que podrían faltar conjuntamente en determinados registros, aunque todavía debemos comprobarlo.

Tercera: las columnas fundamentales para nuestro análisis (track_title, entry_date, stream_duration_ms y product_type) no presentan valores faltantes. Eso es una buena señal para comenzar.

# ¿Los registros sin album_id son exactamente los mismos que no tienen album_title?

In [17]:
sin_album_igual = (df["album_id"].isna() & df["album_title"].isna()).sum()

print(f"Registros donde falta id y titulo de album: {sin_album_igual}")

Registros donde falta id y titulo de album: 1142


Comprobamos que los registros sin album_id coinciden con aquellos que tampoco tienen album_title.

Ahora surge una nueva pregunta: **¿es necesario eliminar estos 1,142 registros o podemos aplicar algún criterio para recuperar la información faltante?**

Antes de tomar una decisión, planteamos las siguientes hipótesis:

1. ¿Es posible que estos registros pertenezcan a álbumes que fueron eliminados del catálogo de TIDAL?

2. Recordemos que TIDAL ofrece tanto música como videos musicales. ¿Los videos se incluyen dentro de los álbumes o se contabilizan por separado?

3. ¿Podríamos completar los valores faltantes utilizando otros registros del dataset que coincidan en el nombre de la canción y el artista?

**Conclusión provisional:** necesitamos continuar explorando los datos antes de decidir cómo tratar estos registros. Por ahora, no contamos con evidencia suficiente para eliminarlos ni para completar automáticamente sus valores faltantes.